# RecurrentWSPR Training & Optuna Optimization

This notebook trains and optimizes the **RecurrentWSPR** model using Optuna hyperparameter tuning.

## Supported Asset Classes
- **Livestock**: LE, HE, GF (suffix: `0830_1200_2`, session end: 13:00 CST)
- **Global Commodities**: CL, NG, GC, SI, HG (suffix: `0700_1000`)

## Model Architecture
RecurrentWSPR processes data through five parallel paths:
1. **Windowed Summary**: LSTM over summary embeddings across W days
2. **Windowed Profile**: LSTM over profile embeddings across W days  
3. **Recent Raster**: RasterResNet on most recent day's rasterized VPIN
4. **Recent Sequential**: IntradayRNN on most recent day's intraday sequence
5. **Recent Spatial Fusion**: Fused profile + raster for current market state

## Optimization Strategy
- Small trial count (<20) for efficiency
- Robust parameter ranges based on prior experiments
- TPE sampler with early pruning

## 1. Environment Setup

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab - skipping drive mount")

In [ ]:
# Add CTAFlow to path (Colab only)
if IN_COLAB:
    import sys
    %cd /content/drive/MyDrive/CTAEnv/
    sys.path.insert(0, '/content/drive/MyDrive/CTAEnv/CTAFlow/')
    sys.path.insert(1, '/content/drive/MyDrive/CTAEnv/SierraPy')
    !pip install -e SierraPy -q
    !pip install -e CTAFlow -q
    !pip install optuna -q
else:
    print("Running locally - ensure CTAFlow and optuna are installed")

In [ ]:

import json
import warnings
from datetime import time, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Optuna version: {optuna.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set seeds for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 2. Asset Class Configuration

In [ ]:
# Asset class configurations
ASSET_CONFIGS = {
    # Livestock futures
    'livestock': {
        'tickers': ['LE_F', 'HE_F', 'GF_F'],
        'data_suffix': '0830_1200_2',
        'session_end': time(13, 0),
        'session_open': time(8, 30),
        'target_time': time(13, 0),
        'target_period': timedelta(minutes=60),
        'tz': 'America/Chicago',
    },
    # Global commodities
    'global': {
        'tickers': ['CL_F', 'NG_F', 'GC_F', 'SI_F', 'HG_F'],
        'data_suffix': '0700_1000',
        'session_end': time(14, 30),
        'session_open': time(7, 0),
        'target_time': time(14, 30),
        'target_period': timedelta(minutes=60),
        'tz': 'America/Chicago',
    },
}

# Select asset class and ticker
ASSET_CLASS = 'livestock'  # 'livestock' or 'global'
TICKER = 'LE_F'  # Select from tickers list above

config = ASSET_CONFIGS[ASSET_CLASS]
print(f"Asset Class: {ASSET_CLASS}")
print(f"Ticker: {TICKER}")
print(f"Data suffix: {config['data_suffix']}")
print(f"Session: {config['session_open']} - {config['session_end']}")
print(f"Target time: {config['target_time']}")

In [ ]:
# Path configuration
if IN_COLAB:
    DRIVE_PATH = Path('/content/drive/MyDrive')
    DATA_PATH = DRIVE_PATH / 'intraday' / 'CSV'
else:
    DRIVE_PATH = Path.cwd().parent
    DATA_PATH = DRIVE_PATH / 'data' / 'intraday'

FEATURES_PATH = DRIVE_PATH / 'features' / TICKER
RESULTS_PATH = DRIVE_PATH / 'results' / TICKER / 'wspr_optuna'
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Build file paths based on asset class
suffix = config['data_suffix']
INTRADAY_PATH = DATA_PATH / f'{TICKER}_5min.csv'
FEATURES_FILE = FEATURES_PATH / f'{TICKER}_features_{suffix}.csv'
SEQUENTIAL_PATH = FEATURES_PATH / f'{TICKER}_session_{suffix}_64bin_vpin.parquet'
PROFILE_PATH = FEATURES_PATH / f'{TICKER}_session_{suffix}_64bin_profiles.npz'
RASTERIZED_PATH = FEATURES_PATH / 'rasterized_vpin.npz'
TARGET_PATH = FEATURES_PATH / f'{TICKER}_{suffix}_target.csv'

print(f"\nData paths:")
print(f"  Features: {FEATURES_PATH}")
print(f"  Results: {RESULTS_PATH}")

In [ ]:
# Verify files exist
print("\nChecking data files...")
required_files = [
    (INTRADAY_PATH, "Intraday OHLCV"),
    (FEATURES_FILE, "Summary features"),
    (SEQUENTIAL_PATH, "Sequential VPIN"),
    (PROFILE_PATH, "Profile arrays"),
    (RASTERIZED_PATH, "Rasterized VPIN"),
]

missing = []
for fpath, name in required_files:
    if fpath.exists():
        print(f"  [OK] {name}")
    else:
        print(f"  [MISSING] {name}: {fpath}")
        missing.append(name)

if missing:
    print(f"\n[WARNING] {len(missing)} files missing - data loading may fail")

## 3. Load & Preprocess Data

In [ ]:
from CTAFlow.models import DeepIDMomentum

# Load data
model_data = DeepIDMomentum.from_files(
    intraday_path=str(INTRADAY_PATH),
    features_path=str(FEATURES_FILE),
    sequential_path=str(SEQUENTIAL_PATH),
    profile_path=str(PROFILE_PATH),
    rasterized_path=str(RASTERIZED_PATH),
    target_path=str(TARGET_PATH) if TARGET_PATH.exists() else None,
    target_col="target",
)

print("Data loaded successfully!")
print(f"  Summary shape: {model_data.training_data['summary'].shape}")
print(f"  Sequential shape: {model_data.sequential_data.shape}")
print(f"  Profile shape: {model_data.profile_array.shape}")
print(f"  Rasterized dates: {len(model_data.rasterized_data)}")

In [ ]:
# Task configuration
TASK = 'classification'  # 'regression' or 'classification'
NUM_CLASSES = 3

# Calculate target
model_data.calculate_target(
    target_time_end=config['target_time'],
    period_length=config['target_period'],
    make_clf=(TASK == 'classification'),
)

# Normalize features
model_data.scale_summary_data(rolling_window=252)
model_data.normalize_sequential_features(
    scale_to_basis_points=True,
    scale_orderflow=True
)

# Get dimensions
dims = model_data.dims
print(f"\nFeature dimensions:")
print(f"  {dims}")

In [ ]:
# Visualize target distribution
if TASK == 'classification':
    fig, ax = plt.subplots(figsize=(8, 5))
    target_counts = model_data.target_data.value_counts().sort_index()
    bars = ax.bar(['Down (0)', 'Neutral (1)', 'Up (2)'][:len(target_counts)], 
                  target_counts.values, color=['#e74c3c', '#95a5a6', '#27ae60'])
    ax.set_ylabel('Count')
    ax.set_title(f'{TICKER} Target Class Distribution')
    for bar, count in zip(bars, target_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
                f'{count}\n({count/len(model_data.target_data)*100:.1f}%)', 
                ha='center', va='bottom')
    plt.tight_layout()
    plt.show()
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(model_data.target_data.dropna(), bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(0, color='red', linestyle='--', label='Zero')
    ax.set_xlabel('Return')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{TICKER} Target Return Distribution')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 4. Define Optuna Objective

In [ ]:
from CTAFlow.models.deep_learning.multi_branch.dual_model import RecurrentWSPR
from CTAFlow.models.deep_learning.training.loss import ExpectedPnLLoss

def create_dataloaders(
    model_data,
    window_size: int,
    batch_size: int,
    val_split: float = 0.2,
):
    """Create windowed dataloaders for RecurrentWSPR."""
    train_loader, val_loader = model_data.get_loaders(
        val_split=True,
        val_split_size=val_split,
        batch_size=batch_size,
        shuffle_train=True,
        num_workers=0,
        windowed=True,
        window_days=window_size,
        use_rasterized=True,
        add_raw_returns=True,
        verbose=False,
    )
    return train_loader, val_loader


def compute_class_weights(targets: np.ndarray) -> torch.Tensor:
    """Compute inverse frequency class weights."""
    classes, counts = np.unique(targets[~np.isnan(targets)], return_counts=True)
    weights = 1.0 / counts
    weights = weights / weights.sum() * len(classes)
    return torch.tensor(weights, dtype=torch.float32)


def train_epoch(model, loader, criterion, optimizer, device, task):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for batch in loader:
        summary_days, seq_days, seq_lens, profile_days, raster_days, targets, raw_returns = batch
        
        # Move to device
        summary_days = summary_days.to(device)
        profile_days = profile_days.to(device)
        raster_recent = raster_days[:, -1].to(device)
        seq_recent = seq_days[:, -1].to(device)
        seq_lens_recent = seq_lens[:, -1].to(device)
        targets = targets.to(device)
        raw_returns = raw_returns.to(device)
        
        if task == 'classification':
            targets = targets.long()
        
        optimizer.zero_grad()
        outputs = model(summary_days, profile_days, raster_recent, seq_recent, seq_lens_recent)
        
        if task == 'classification':
            loss = criterion(outputs, targets, returns=raw_returns)
            _, predicted = outputs.max(1)
            correct += predicted.eq(targets).sum().item()
        else:
            loss = criterion(outputs.squeeze(), targets)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        total += targets.size(0)
    
    avg_loss = total_loss / len(loader)
    accuracy = 100.0 * correct / total if task == 'classification' else None
    return avg_loss, accuracy


def evaluate(model, loader, criterion, device, task):
    """Evaluate model on validation set."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in loader:
            summary_days, seq_days, seq_lens, profile_days, raster_days, targets, raw_returns = batch
            
            summary_days = summary_days.to(device)
            profile_days = profile_days.to(device)
            raster_recent = raster_days[:, -1].to(device)
            seq_recent = seq_days[:, -1].to(device)
            seq_lens_recent = seq_lens[:, -1].to(device)
            targets = targets.to(device)
            raw_returns = raw_returns.to(device)
            
            if task == 'classification':
                targets = targets.long()
            
            outputs = model(summary_days, profile_days, raster_recent, seq_recent, seq_lens_recent)
            
            if task == 'classification':
                loss = criterion(outputs, targets, returns=raw_returns)
                _, predicted = outputs.max(1)
                correct += predicted.eq(targets).sum().item()
            else:
                loss = criterion(outputs.squeeze(), targets)
            
            total_loss += loss.item()
            total += targets.size(0)
    
    avg_loss = total_loss / len(loader)
    accuracy = 100.0 * correct / total if task == 'classification' else None
    return avg_loss, accuracy


print("Training utilities defined.")

In [ ]:
def objective(trial: optuna.Trial) -> float:
    """
    Optuna objective function for RecurrentWSPR optimization.
    
    Optimizes:
    - Architecture: d_model, lstm hidden sizes
    - Training: learning rate, weight decay, batch size
    - Regularization: dropout
    - Window size
    """
    # Hyperparameters to optimize (robust ranges)
    window_size = trial.suggest_int('window_size', 5, 15)
    d_model = trial.suggest_categorical('d_model', [64, 128, 256])
    sum_lstm_hidden = trial.suggest_categorical('sum_lstm_hidden', [32, 64, 128])
    prof_lstm_hidden = trial.suggest_categorical('prof_lstm_hidden', [64, 128, 256])
    dropout = trial.suggest_float('dropout', 0.1, 0.4, step=0.1)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-4, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    ce_weight = trial.suggest_float('ce_weight', 0.05, 0.5, log=True)
    transaction_cost = trial.suggest_float('transaction_cost', 0.0, 0.001, log=True)
    temperature = trial.suggest_float('temperature', 0.5, 2.0)
    
    # Create dataloaders
    try:
        train_loader, val_loader = create_dataloaders(
            model_data,
            window_size=window_size,
            batch_size=batch_size,
            val_split=0.2,
        )
    except Exception as e:
        print(f"Dataloader creation failed: {e}")
        return float('inf') if TASK == 'regression' else 0.0
    
    # Create model
    model = RecurrentWSPR(
        f_sum=dims.summary_dim,
        f_profile=dims.profile_channels,
        f_raster=dims.raster_channels,
        f_seq=dims.seq_dim,
        d_model=d_model,
        sum_lstm_hidden=sum_lstm_hidden,
        prof_lstm_hidden=prof_lstm_hidden,
        task=TASK,
        num_classes=NUM_CLASSES if TASK == 'classification' else 1,
        dropout=dropout,
    ).to(device)
    
    # Loss and optimizer
    if TASK == 'classification':
        criterion = ExpectedPnLLoss(
            ce_weight=ce_weight,
            transaction_cost=transaction_cost,
            temperature=temperature
        )
    else:
        criterion = nn.MSELoss()
    
    optimizer = optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )
    
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=20, eta_min=learning_rate * 0.01
    )
    
    # Training loop with early stopping
    best_metric = float('-inf')
    patience = 5
    patience_counter = 0
    
    for epoch in range(20):  # Max 20 epochs per trial
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, TASK)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device, TASK)
        scheduler.step()
        
        # The objective is to maximize PnL, which means minimizing the PnL loss.
        # The loss is -PnL + ce_loss, so we want to minimize it.
        # Optuna maximizes, so we return the negative validation loss.
        current_metric = -val_loss
        improved = current_metric > best_metric
        
        if improved:
            best_metric = current_metric
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Report to Optuna for pruning
        trial.report(current_metric, epoch)
        
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        if patience_counter >= patience:
            break
    
    # Return metric (higher is better)
    return best_metric


print("Objective function defined.")

## 5. Run Optuna Optimization

In [ ]:
# Optimization settings
N_TRIALS = 15  # Keep small for efficiency
STUDY_NAME = f"{TICKER}_wspr_{TASK}"

# Create study
direction = 'maximize' if TASK == 'classification' else 'minimize'

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction=direction,
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=5),
)

print(f"Starting optimization: {N_TRIALS} trials")
print(f"Study name: {STUDY_NAME}")
print(f"Direction: {direction}")
print(f"Task: {TASK}")
print("-" * 50)

In [ ]:
# Run optimization
study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
    gc_after_trial=True,
)

In [ ]:
# Display results
print("\n" + "=" * 60)
print("OPTIMIZATION COMPLETE")
print("=" * 60)

print(f"\nBest trial:")
trial = study.best_trial
print(f"  Value: {trial.value:.4f}")
print(f"  Params:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

# Save best params
best_params = trial.params.copy()
best_params['best_value'] = trial.value
best_params['ticker'] = TICKER
best_params['task'] = TASK

with open(RESULTS_PATH / f'{TICKER}_best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
print(f"\nBest params saved to: {RESULTS_PATH / f'{TICKER}_best_params.json'}")

## 6. Visualization

In [ ]:
# Optimization history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Optimization history
ax = axes[0, 0]
trials_df = study.trials_dataframe()
valid_trials = trials_df[trials_df['state'] == 'COMPLETE']
ax.plot(valid_trials.index, valid_trials['value'], 'b-o', alpha=0.6, label='Trial value')
ax.axhline(y=study.best_value, color='r', linestyle='--', label=f'Best: {study.best_value:.4f}')
ax.set_xlabel('Trial')
ax.set_ylabel('Accuracy (%)' if TASK == 'classification' else 'Negative MSE')
ax.set_title('Optimization History')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Parameter importance
ax = axes[0, 1]
try:
    importances = optuna.importance.get_param_importances(study)
    params = list(importances.keys())
    values = list(importances.values())
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(params)))
    bars = ax.barh(params, values, color=colors)
    ax.set_xlabel('Importance')
    ax.set_title('Hyperparameter Importance')
    ax.grid(True, alpha=0.3, axis='x')
except:
    ax.text(0.5, 0.5, 'Not enough trials\nfor importance analysis', 
            ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Hyperparameter Importance')

# 3. Learning rate vs performance
ax = axes[1, 0]
if 'params_learning_rate' in valid_trials.columns:
    ax.scatter(valid_trials['params_learning_rate'], valid_trials['value'], 
               c=valid_trials.index, cmap='viridis', alpha=0.7, s=100)
    ax.set_xscale('log')
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('Performance')
    ax.set_title('Learning Rate vs Performance')
    ax.grid(True, alpha=0.3)

# 4. d_model vs performance
ax = axes[1, 1]
if 'params_d_model' in valid_trials.columns:
    d_models = valid_trials['params_d_model'].unique()
    positions = range(len(d_models))
    data_by_d = [valid_trials[valid_trials['params_d_model'] == d]['value'].values for d in d_models]
    bp = ax.boxplot(data_by_d, positions=positions, patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.Set2(np.linspace(0, 1, len(d_models)))):
        patch.set_facecolor(color)
    ax.set_xticks(positions)
    ax.set_xticklabels([str(int(d)) for d in d_models])
    ax.set_xlabel('d_model')
    ax.set_ylabel('Performance')
    ax.set_title('Model Size vs Performance')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'{TICKER} RecurrentWSPR Optimization Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f'{TICKER}_optimization_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Parallel coordinate plot
fig = optuna.visualization.plot_parallel_coordinate(
    study, 
    params=['window_size', 'd_model', 'learning_rate', 'dropout']
)
fig.update_layout(title=f'{TICKER} Parallel Coordinate Plot')
fig.write_image(str(RESULTS_PATH / f'{TICKER}_parallel_coords.png'))
fig.show()

## 7. Train Final Model with Best Parameters

In [ ]:
# Extract best parameters
best = study.best_params

print("Training final model with best parameters:")
for k, v in best.items():
    print(f"  {k}: {v}")

In [ ]:
# Create final dataloaders
train_loader, val_loader = create_dataloaders(
    model_data,
    window_size=best['window_size'],
    batch_size=best['batch_size'],
    val_split=0.2,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

In [ ]:
# Create final model
final_model = RecurrentWSPR(
    f_sum=dims.summary_dim,
    f_profile=dims.profile_channels,
    f_raster=dims.raster_channels,
    f_seq=dims.seq_dim,
    d_model=best['d_model'],
    sum_lstm_hidden=best['sum_lstm_hidden'],
    prof_lstm_hidden=best['prof_lstm_hidden'],
    task=TASK,
    num_classes=NUM_CLASSES if TASK == 'classification' else 1,
    dropout=best['dropout'],
).to(device)

print(f"\nModel parameters: {sum(p.numel() for p in final_model.parameters()):,}")

In [ ]:
# Training settings
NUM_EPOCHS = 50

if TASK == 'classification':
    class_weights = compute_class_weights(model_data.target_data.values).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.MSELoss()

optimizer = optim.AdamW(
    final_model.parameters(),
    lr=best['learning_rate'],
    weight_decay=best['weight_decay']
)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=best['learning_rate'] * 0.001
)

# Training history
history = {
    'train_loss': [], 'val_loss': [],
    'train_metric': [], 'val_metric': [],
    'lr': []
}

best_metric = float('-inf') if TASK == 'classification' else float('inf')
best_state = None

print(f"\nStarting final training for {NUM_EPOCHS} epochs...")
print("=" * 60)

In [ ]:
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(final_model, train_loader, criterion, optimizer, device, TASK)
    val_loss, val_acc = evaluate(final_model, val_loader, criterion, device, TASK)
    
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(current_lr)
    
    if TASK == 'classification':
        history['train_metric'].append(train_acc)
        history['val_metric'].append(val_acc)
        
        if val_acc > best_metric:
            best_metric = val_acc
            best_state = final_model.state_dict().copy()
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | Loss: {train_loss:.4f}/{val_loss:.4f} | "
                  f"Acc: {train_acc:.2f}%/{val_acc:.2f}% | LR: {current_lr:.2e}")
    else:
        history['train_metric'].append(train_loss)
        history['val_metric'].append(val_loss)
        
        if val_loss < best_metric:
            best_metric = val_loss
            best_state = final_model.state_dict().copy()
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | Loss: {train_loss:.6f}/{val_loss:.6f} | LR: {current_lr:.2e}")

print("\n" + "=" * 60)
if TASK == 'classification':
    print(f"Best validation accuracy: {best_metric:.2f}%")
else:
    print(f"Best validation loss: {best_metric:.6f}")

# Load best model
final_model.load_state_dict(best_state)

In [ ]:
# Plot final training history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
ax = axes[0]
ax.plot(history['train_loss'], label='Train', alpha=0.8)
ax.plot(history['val_loss'], label='Validation', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training & Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Metric
ax = axes[1]
ax.plot(history['train_metric'], label='Train', alpha=0.8)
ax.plot(history['val_metric'], label='Validation', alpha=0.8)
ax.set_xlabel('Epoch')
ylabel = 'Accuracy (%)' if TASK == 'classification' else 'Loss'
ax.set_ylabel(ylabel)
ax.set_title(f'Training & Validation {ylabel}')
ax.legend()
ax.grid(True, alpha=0.3)

# Learning rate
ax = axes[2]
ax.plot(history['lr'], 'g-', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

plt.suptitle(f'{TICKER} RecurrentWSPR Final Training', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f'{TICKER}_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Final Evaluation & Predictions

In [ ]:
# Collect all predictions
final_model.eval()
all_preds = []
all_targets = []
all_probs = []

with torch.no_grad():
    for batch in val_loader:
        summary_days, seq_days, seq_lens, profile_days, raster_days, targets = batch
        
        summary_days = summary_days.to(device)
        profile_days = profile_days.to(device)
        raster_recent = raster_days[:, -1].to(device)
        seq_recent = seq_days[:, -1].to(device)
        seq_lens_recent = seq_lens[:, -1].to(device)
        
        outputs = final_model(summary_days, profile_days, raster_recent, seq_recent, seq_lens_recent)
        
        if TASK == 'classification':
            probs = torch.softmax(outputs, dim=1)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
        else:
            all_preds.extend(outputs.squeeze().cpu().numpy())
        
        all_targets.extend(targets.numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
if TASK == 'classification':
    all_probs = np.array(all_probs)

print(f"Predictions: {len(all_preds)}")
print(f"Targets: {len(all_targets)}")

In [ ]:
if TASK == 'classification':
    from sklearn.metrics import classification_report, confusion_matrix
    
    # Classification report
    print("\nClassification Report:")
    print("=" * 50)
    labels = ['Down', 'Neutral', 'Up'][:NUM_CLASSES]
    print(classification_report(all_targets.astype(int), all_preds, target_names=labels))
    
    # Confusion matrix
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Raw counts
    cm = confusion_matrix(all_targets.astype(int), all_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=labels, yticklabels=labels)
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    axes[0].set_title('Confusion Matrix (Counts)')
    
    # Normalized
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
                xticklabels=labels, yticklabels=labels)
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')
    axes[1].set_title('Confusion Matrix (Normalized)')
    
    plt.suptitle(f'{TICKER} RecurrentWSPR Confusion Matrix', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_PATH / f'{TICKER}_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    
else:
    # Regression evaluation
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    
    mse = mean_squared_error(all_targets, all_preds)
    mae = mean_absolute_error(all_targets, all_preds)
    r2 = r2_score(all_targets, all_preds)
    corr = np.corrcoef(all_targets, all_preds)[0, 1]
    
    print(f"\nRegression Metrics:")
    print(f"  MSE: {mse:.6f}")
    print(f"  MAE: {mae:.6f}")
    print(f"  R²: {r2:.4f}")
    print(f"  Correlation: {corr:.4f}")
    
    # Scatter plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    ax = axes[0]
    ax.scatter(all_targets, all_preds, alpha=0.5, s=20)
    lims = [min(all_targets.min(), all_preds.min()), max(all_targets.max(), all_preds.max())]
    ax.plot(lims, lims, 'r--', lw=2, label='Perfect')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(f'Predictions vs Actual (Corr: {corr:.4f})')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Residuals
    ax = axes[1]
    residuals = all_preds - all_targets
    ax.hist(residuals, bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(0, color='red', linestyle='--')
    ax.set_xlabel('Residual')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Residual Distribution (MAE: {mae:.6f})')
    ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'{TICKER} RecurrentWSPR Regression Results', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_PATH / f'{TICKER}_regression_results.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. Save Final Model

In [ ]:
# Save model
model_path = RESULTS_PATH / f'{TICKER}_wspr_best.pth'
torch.save({
    'model_state_dict': final_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_metric': best_metric,
    'params': best,
    'dims': {
        'summary_dim': dims.summary_dim,
        'seq_dim': dims.seq_dim,
        'profile_shape': dims.profile_shape,
        'raster_shape': dims.raster_shape,
    },
    'task': TASK,
    'num_classes': NUM_CLASSES,
}, model_path)

print(f"Model saved to: {model_path}")

# Save training history
history_df = pd.DataFrame(history)
history_df.to_csv(RESULTS_PATH / f'{TICKER}_training_history.csv', index=False)

# Summary
print(f"\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"\nArtifacts saved to: {RESULTS_PATH}")
print(f"  - {TICKER}_wspr_best.pth (model checkpoint)")
print(f"  - {TICKER}_best_params.json (best hyperparameters)")
print(f"  - {TICKER}_training_history.csv")
print(f"  - {TICKER}_optimization_results.png")
print(f"  - {TICKER}_training_history.png")
if TASK == 'classification':
    print(f"  - {TICKER}_confusion_matrix.png")
else:
    print(f"  - {TICKER}_regression_results.png")